# Track 02 — Tối ưu tốc độ trên Kaggle: đo, không đoán

**Thước đo:** thời gian để train xong **cùng một lượng dữ liệu** (400 mẫu, 1 epoch).

Đây là điểm phương pháp luận quan trọng: **không được so `giây/step`** khi bật `packing`,
vì packing thay đổi lượng dữ liệu mỗi step. Chỉ "thời gian xử lý xong cùng tập dữ liệu"
là so sánh công bằng.

### Các cần điều khiển được đo

| Mục | So gì | Vì sao |
|---|---|---|
| 1 | **bf16 (emulation) vs fp16** | Định lượng đúng cái bug đã làm run đầu chạy 623 giây |
| 2 | padding động vs `group_by_length` vs `packing` | Dữ liệu có p50=124 token nhưng max_len 512 → thừa rất nhiều |
| 3 | batch 8 / 16 / 32 | VRAM đỉnh chỉ 12.75/15.6 GB → còn dư |
| 4 | `eager` vs `sdpa` | FlashAttention-2 **không chạy trên T4** (cần sm_80+) |
| 5 | 1×T4 vs **2×T4 data parallel** | "Chạy hết công suất" |

## 0. Cài + môi trường

In [ ]:
# ================= GHIM 1 GPU - PHAI CHAY TRUOC MOI import torch =================
# Kaggle cap 2xT4. Khi thay NHIEU GPU, transformers Trainer AM THAM boc model vao
# nn.DataParallel -> batch_size thuc te nhan doi; TRL SFTTrainer thi tu choi thang.
import os, subprocess
_n = subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout.strip().splitlines()
print(f"May nay co {len(_n)} GPU:")
for l in _n: print("  ", l)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Giam phan manh bo nho giua cac lan bench: run truoc OOM o 14.56GB du VRAM dinh
# cua cau hinh nhanh nhat chi 7.61GB -> phan manh chu khong phai thieu that.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("-> Ghim CUDA_VISIBLE_DEVICES=0 cho MOI benchmark trong notebook (single-GPU, cong bang).")
print("   Muc 6 truyen CUDA_VISIBLE_DEVICES=0,1 RIENG cho subprocess accelerate launch.")
N_GPU_PHYSICAL = len(_n)

In [ ]:
!pip install -q -U "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "datasets==5.1.0" accelerate
!pip uninstall -q -y torchao 2>/dev/null; echo "da go torchao (xung dot voi peft 0.21.2)"

In [ ]:
import torch, transformers, time, json, gc, os, subprocess, sys
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

N_GPU = torch.cuda.device_count()
CAP   = torch.cuda.get_device_capability(0)
GPU   = torch.cuda.get_device_name(0)
BF16_NATIVE = CAP[0] >= 8
print(f"GPU torch thay: {GPU} x{N_GPU} (vat ly: {N_GPU_PHYSICAL}) | cap {CAP}")
print(f"torch.cuda.is_bf16_supported() bao: {torch.cuda.is_bf16_supported()}")
print(f"bf16 NATIVE (cap>=8.0)            : {BF16_NATIVE}")
if not BF16_NATIVE:
    print("-> Card nay KHONG co bf16 native. Muc 1 se do gia phai tra cua bf16 emulation.")
else:
    print("-> Card nay CO bf16 native. Muc 1 se do bf16 vs fp16 that su.")
for i in range(N_GPU):
    print(f"   GPU{i}: {torch.cuda.get_device_name(i)} "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")

In [ ]:
import matplotlib as mpl, matplotlib.pyplot as plt, numpy as np
SURFACE="#fcfcfb"; INK="#0b0b0b"; INK2="#52514e"; GRID="#e5e4e0"
S1="#2a78d6"; S2="#eb6834"; S3="#1baf7a"
mpl.rcParams.update({"figure.facecolor":SURFACE,"axes.facecolor":SURFACE,"axes.edgecolor":GRID,
 "axes.labelcolor":INK2,"text.color":INK,"xtick.color":INK2,"ytick.color":INK2,
 "axes.spines.top":False,"axes.spines.right":False,"grid.color":GRID,"grid.linewidth":.8,
 "font.size":10,"axes.titlesize":12,"lines.linewidth":2,"lines.markersize":6,"figure.dpi":120})
def finish(ax,title,ylab=None,xlab=None,legend=False):
    ax.set_title(title,loc="left",pad=12,color=INK)
    if ylab: ax.set_ylabel(ylab)
    if xlab: ax.set_xlabel(xlab)
    ax.grid(axis="y",alpha=.7); ax.set_axisbelow(True)
    if legend: ax.legend(frameon=False)
    plt.tight_layout()
def barchart(names, vals, title, unit, fmt="{:.1f}", colors=None, lower_better=True):
    fig,ax=plt.subplots(figsize=(max(5.0,1.35*len(names)),3.6))
    cols = colors or [S1]*len(names)
    best = (min if lower_better else max)(vals)
    cols = [S3 if v==best else c for v,c in zip(vals,cols)]
    bars=ax.bar(names,vals,width=.55,color=cols,edgecolor=SURFACE,linewidth=2)
    for b,v in zip(bars,vals):
        ax.annotate(fmt.format(v),(b.get_x()+b.get_width()/2,v),textcoords="offset points",
                    xytext=(0,5),ha="center",color=INK,fontsize=10,fontweight="600")
    ax.set_ylim(0,max(vals)*1.25); finish(ax,title,unit)
    plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

print("style ok")

## 1. Dữ liệu + hạ tầng đo

400 mẫu, 1 epoch. Mỗi cấu hình train lại từ model sạch để không lẫn trạng thái.

In [ ]:
MODEL="Qwen/Qwen3-0.6B"; N=400; MAX_LEN=512
SYSTEM="Ban la bo dich text-to-SQL. Chi tra ve cau truy van SQLite."
tok=AutoTokenizer.from_pretrained(MODEL)
raw=load_dataset("b-mc2/sql-create-context",split="train").shuffle(seed=42).select(range(N))

def prompt_of(ex):
    return tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},
         {"role":"user","content":f"Schema:\n{ex['context']}\n\nCau hoi: {ex['question']}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

def feat(ex):
    p=tok(prompt_of(ex),add_special_tokens=False)["input_ids"]
    a=tok(ex["answer"]+tok.eos_token,add_special_tokens=False)["input_ids"]
    ids=(p+a)[:MAX_LEN]
    return {"input_ids":ids,"labels":([-100]*len(p)+a)[:MAX_LEN],"length":len(ids)}

DS  = raw.map(feat, remove_columns=raw.column_names, desc="tokenize")
LENS= np.array(DS["length"])
print(f"do dai token: p50={np.percentile(LENS,50):.0f} p95={np.percentile(LENS,95):.0f} max={LENS.max()}")
print(f"-> pad den max_len={MAX_LEN} se bo di ~{100*(1-LENS.mean()/MAX_LEN):.0f}% tinh toan")
print(f"-> pad dong (theo batch-max) tiet kiem phan lon so do")

def coll(fs):
    n=max(len(f["input_ids"]) for f in fs); i=[];l=[];a=[]
    for f in fs:
        k=n-len(f["input_ids"])
        i.append(f["input_ids"]+[tok.pad_token_id]*k); l.append(f["labels"]+[-100]*k)
        a.append([1]*len(f["input_ids"])+[0]*k)
    t=torch.tensor; return {"input_ids":t(i),"labels":t(l),"attention_mask":t(a)}

LORA=dict(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",
          target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def clear():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# ===================== PHONG VE TRUOC THAY DOI API =====================
# BAI HOC THAT: run dau cua notebook nay ERROR vi
#   TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'group_by_length'
# transformers v5 da bo `group_by_length` (giong nhu da bo `warmup_ratio`).
# Thay vi doan, HAY INTROSPECT. Helper duoi tu loc tham so khong ton tai.
import inspect
TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
print(f"TrainingArguments ban {transformers.__version__} co {len(TA_PARAMS)} tham so\n")
print("Cac can dieu khien toc do - ban nay co khong?")
for p in ["group_by_length","length_column_name","warmup_ratio","warmup_steps",
          "gradient_checkpointing","optim","torch_compile","dataloader_num_workers",
          "bf16","fp16","per_device_train_batch_size","gradient_accumulation_steps"]:
    print(f"  {p:30s}", "CO" if p in TA_PARAMS else "KHONG CO o ban nay")

def ta(**kw):
    'Tao TrainingArguments, bo im lang tham so ban nay khong ho tro.'
    miss = [k for k in kw if k not in TA_PARAMS]
    if miss: print("   [bo tham so khong ton tai]:", miss)
    return TrainingArguments(**{k:v for k,v in kw.items() if k in TA_PARAMS})

HAS_GBL = "group_by_length" in TA_PARAMS

RESULTS=[]
def bench(label, *, dtype=None, bs=8, accum=1, group_by_length=False,
          attn=None, ds=None, extra=None):
    """Train 1 epoch tren cung 400 mau. Tra ve giay + VRAM dinh."""
    clear()
    dt = dtype if dtype is not None else (torch.bfloat16 if BF16_NATIVE else torch.float16)
    kw = {}
    if attn: kw["attn_implementation"]=attn
    m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dt, device_map="cuda", **kw)
    m.config.use_cache=False
    m = get_peft_model(m, LoraConfig(task_type="CAUSAL_LM", **LORA))
    args = dict(output_dir="/kaggle/working/_b", per_device_train_batch_size=bs,
                gradient_accumulation_steps=accum, num_train_epochs=1,
                learning_rate=2e-4, logging_steps=1000, save_strategy="no", report_to=[],
                bf16=(dt==torch.bfloat16), fp16=(dt==torch.float16),
                group_by_length=group_by_length, seed=0)
    if group_by_length: args["length_column_name"]="length"
    if extra: args.update(extra)
    tr = Trainer(model=m, args=ta(**args),
                 train_dataset=(ds if ds is not None else DS), data_collator=coll)
    t0=time.time(); out=tr.train(); secs=time.time()-t0
    peak=torch.cuda.max_memory_allocated()/1e9
    r={"name":label,"secs":secs,"vram":peak,"loss":out.training_loss,
       "dtype":str(dt).replace("torch.",""),"bs":bs}
    RESULTS.append(r); del m,tr; clear()
    print(f"{label:30s} {secs:6.1f}s | VRAM {peak:5.2f}GB | loss {out.training_loss:.4f}")
    return r
print("\nha tang do san sang")

## 2. 📊 bf16 emulation vs fp16 — định lượng cái bug

Trên T4 (sm_75) `bf16` chạy bằng **emulation**, không chạm tensor core. Trên Ampere+ thì
bf16 là native và kết quả sẽ khác hẳn. Ô này đo đúng cái giá đó.

In [ ]:
bench("fp16", dtype=torch.float16)
bench("bf16", dtype=torch.bfloat16)

f16=[r for r in RESULTS if r["name"]=="fp16"][0]
b16=[r for r in RESULTS if r["name"]=="bf16"][0]
barchart(["fp16","bf16"],[f16["secs"],b16["secs"]],
         "Thoi gian train 1 epoch / 400 mau","giay","{:.1f}s")
ratio=b16["secs"]/f16["secs"]
print(f"bf16 / fp16 = {ratio:.2f}x")
if not BF16_NATIVE:
    print(f"-> Tren {GPU} (khong co bf16 native): bf16 cham hon fp16 {ratio:.2f}x.")
    print(f"   Day la gia phai tra cua bug auto-detect o run dau tien.")
else:
    print(f"-> Tren {GPU} co bf16 native nen hai cai sat nhau; chon bf16 vi on dinh so hoc tot hon.")
print(f"\n{'Cau hinh':12s}{'Giay':>9s}{'VRAM GB':>10s}")
print("-"*31)
for r in (f16,b16): print(f"{r['name']:12s}{r['secs']:>9.1f}{r['vram']:>10.2f}")

## 3. 📊 Chiến lược padding: động vs `group_by_length` vs `packing`

Dữ liệu p50≈124 token. Padding động (collator của ta) đã pad theo batch-max thay vì 512.
`group_by_length` gom mẫu cùng độ dài → batch-max nhỏ hơn nữa.
`packing` (TRL) nhồi nhiều mẫu vào một sequence → gần như không còn padding.

In [ ]:
BEST_DT = torch.float16 if not BF16_NATIVE else torch.bfloat16
bench("pad dong", dtype=BEST_DT)

# group_by_length da bi bo o transformers v5. Nhung HIEU UNG cua no thi tai tao duoc:
# sap xep dataset theo do dai -> cac mau trong cung batch co do dai gan nhau -> it padding.
# Cach nay chay duoc tren MOI ban, khong phu thuoc API.
DS_SORTED = DS.sort("length")
bench("sort theo length", dtype=BEST_DT, ds=DS_SORTED)

if HAS_GBL:
    bench("group_by_length (flag)", dtype=BEST_DT, group_by_length=True)
else:
    print("\n-> Ban transformers nay KHONG co `group_by_length`.")
    print('   Dung `DS.sort("length")` nhu tren de co cung hieu ung.')
    print("   Luu y: sort lam mat tinh ngau nhien thu tu mau -> voi 1 epoch thi anh huong nho,")
    print("   nhieu epoch thi nen shuffle lai moi epoch hoac dung packing.")

In [ ]:
# packing qua TRL SFTTrainer (collator rieng, nen dung SFTTrainer thay vi Trainer)
from trl import SFTTrainer, SFTConfig
def bench_packing(label, packing):
    clear()
    ds = raw.map(lambda ex:{"prompt":prompt_of(ex),"completion":ex["answer"]},
                 remove_columns=raw.column_names)
    m=AutoModelForCausalLM.from_pretrained(MODEL,dtype=BEST_DT,device_map="cuda")
    m.config.use_cache=False
    tr=SFTTrainer(model=m, train_dataset=ds,
        peft_config=LoraConfig(task_type="CAUSAL_LM",**LORA),
        args=SFTConfig(output_dir="/kaggle/working/_p", per_device_train_batch_size=8,
            num_train_epochs=1, learning_rate=2e-4, logging_steps=1000,
            save_strategy="no", report_to=[], max_length=MAX_LEN,
            packing=packing, gradient_checkpointing=False,
            bf16=(BEST_DT==torch.bfloat16), fp16=(BEST_DT==torch.float16), seed=0))
    t0=time.time(); out=tr.train(); secs=time.time()-t0
    peak=torch.cuda.max_memory_allocated()/1e9
    RESULTS.append({"name":label,"secs":secs,"vram":peak,"loss":out.training_loss,
                    "dtype":str(BEST_DT).replace("torch.",""),"bs":8})
    print(f"{label:30s} {secs:6.1f}s | VRAM {peak:5.2f}GB | loss {out.training_loss:.4f}")
    del m,tr; clear()

bench_packing("TRL khong packing", False)
bench_packing("TRL packing=True",  True)

In [ ]:
pad_names=["pad dong","sort theo length","group_by_length (flag)",
           "TRL khong packing","TRL packing=True"]
pad=[r for n in pad_names for r in RESULTS if r["name"]==n]
barchart([r["name"] for r in pad],[r["secs"] for r in pad],
         "Chien luoc padding - thoi gian 1 epoch","giay","{:.1f}s")
base=pad[0]["secs"]
print(f"{'Chien luoc':24s}{'Giay':>8s}{'vs pad dong':>13s}{'VRAM':>8s}")
print("-"*53)
for r in pad:
    print(f"{r['name']:24s}{r['secs']:>8.1f}{base/r['secs']:>12.2f}x{r['vram']:>8.2f}")
print("\nLuu y: loss khac nhau giua packing/khong packing la BINH THUONG -")
print("packing gop nhieu mau vao 1 sequence nen cach tinh trung binh loss khac.")

## 4. 📊 Batch size

VRAM đỉnh run trước chỉ 12.75/15.6 GB → còn dư. Tăng batch thường là cách rẻ nhất.

In [ ]:
for bs in [8,16,32]:
    try: bench(f"batch {bs}", dtype=BEST_DT, bs=bs)
    except torch.cuda.OutOfMemoryError as e:
        # BUG DA GAP: truoc day viet f"batch {bs:<24s}" nhung bs la int
        # -> ValueError: Unknown format code 's' for object of type 'int'
        # Handler loi CUNG phai duoc test.
        print(f"batch {bs}: OOM ({str(e)[:70]})"); clear()
    except Exception as e:
        print(f"batch {bs}: {type(e).__name__}: {str(e)[:80]}"); clear()
bs_r=[r for r in RESULTS if r["name"].startswith("batch ")]
if bs_r:
    barchart([r["name"] for r in bs_r],[r["secs"] for r in bs_r],
             "Batch size - thoi gian 1 epoch","giay","{:.1f}s")
    barchart([r["name"] for r in bs_r],[r["vram"] for r in bs_r],
             "Batch size - VRAM dinh","GB","{:.2f}", lower_better=False)

## 5. 📊 Attention: `eager` vs `sdpa`

⚠️ **FlashAttention-2 không chạy trên T4** — nó cần sm_80 (Ampere) trở lên. Mọi tutorial đều
khuyên FA2 nhưng trên Kaggle T4 là vô dụng. `sdpa` là lựa chọn đúng.

In [ ]:
for attn in ["eager","sdpa"]:
    try: bench(f"attn={attn}", dtype=BEST_DT, attn=attn)
    except Exception as e: print(f"attn={attn}: {type(e).__name__}: {str(e)[:90]}")
at=[r for r in RESULTS if r["name"].startswith("attn=")]
if len(at)>1:
    barchart([r["name"] for r in at],[r["secs"] for r in at],
             "Attention implementation","giay","{:.1f}s")
# FA2 co that su khong dung duoc khong? Thu that.
try:
    import flash_attn; print("flash-attn da cai:", flash_attn.__version__)
except ImportError:
    print("flash-attn chua cai. Tren sm_75 (T4) thi KHONG cai duoc - can sm_80+.")
    print(f"Card nay: compute cap {CAP} ->", "dung duoc FA2" if CAP[0]>=8 else "KHONG dung duoc FA2")

## 6. 📊 Dùng cả 2 GPU — data parallel thật

⚠️ `device_map="auto"` **KHÔNG** làm việc này — nó chia layer (model parallel), model nhỏ thì
chỉ thêm overhead chứ không nhanh hơn. Muốn nhanh hơn phải **data parallel** qua
`accelerate launch --multi_gpu`, và nó cần một **script riêng**, không chạy trong notebook cell.

In [ ]:
SCRIPT = r'''
import torch, time, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from accelerate import Accelerator

MODEL="Qwen/Qwen3-0.6B"; N=400; MAX_LEN=512
SYSTEM="Ban la bo dich text-to-SQL. Chi tra ve cau truy van SQLite."
acc=Accelerator()
tok=AutoTokenizer.from_pretrained(MODEL)
raw=load_dataset("b-mc2/sql-create-context",split="train").shuffle(seed=42).select(range(N))
def prompt_of(ex):
    return tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},
         {"role":"user","content":f"Schema:
{ex['context']}

Cau hoi: {ex['question']}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
def feat(ex):
    p=tok(prompt_of(ex),add_special_tokens=False)["input_ids"]
    a=tok(ex["answer"]+tok.eos_token,add_special_tokens=False)["input_ids"]
    return {"input_ids":(p+a)[:MAX_LEN],"labels":([-100]*len(p)+a)[:MAX_LEN]}
ds=raw.map(feat,remove_columns=raw.column_names)
def coll(fs):
    n=max(len(f["input_ids"]) for f in fs); i=[];l=[];a=[]
    for f in fs:
        k=n-len(f["input_ids"])
        i.append(f["input_ids"]+[tok.pad_token_id]*k); l.append(f["labels"]+[-100]*k)
        a.append([1]*len(f["input_ids"])+[0]*k)
    t=torch.tensor; return {"input_ids":t(i),"labels":t(l),"attention_mask":t(a)}
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
m=get_peft_model(AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16 if BF16 else torch.float16),
    LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
      target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
m.config.use_cache=False
tr=Trainer(model=m, args=TrainingArguments(
    output_dir="/kaggle/working/_dp", per_device_train_batch_size=8,
    num_train_epochs=1, learning_rate=2e-4, logging_steps=1000,
    save_strategy="no", report_to=[], bf16=BF16, fp16=not BF16, seed=0),
    train_dataset=ds, data_collator=coll)
t0=time.time(); tr.train(); secs=time.time()-t0
if acc.is_main_process:
    print(f"RESULT_SECS={secs:.2f} NUM_PROC={acc.num_processes}")
'''
open("/kaggle/working/train_dp.py","w").write(SCRIPT)
print("da viet /kaggle/working/train_dp.py | so GPU thay duoc:", N_GPU)

In [ ]:
import re
dp_secs=None; sp_secs=None
for nproc in ([1,2] if N_GPU_PHYSICAL>1 else [1]):
    cmd=f"cd /kaggle/working && accelerate launch --num_processes={nproc} " \
        + ("--multi_gpu " if nproc>1 else "") + "train_dp.py"
    print("$",cmd)
    env={**os.environ,"CUDA_VISIBLE_DEVICES":",".join(str(i) for i in range(nproc))}
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True,timeout=3600,env=env)
    mm=re.search(r"RESULT_SECS=([\d.]+)",(r.stdout or "")+(r.stderr or ""))
    if mm:
        s=float(mm.group(1)); print(f"   -> {nproc} GPU: {s:.1f}s")
        if nproc==1: sp_secs=s
        else: dp_secs=s
        RESULTS.append({"name":f"{nproc}xGPU (accelerate)","secs":s,"vram":float("nan"),
                        "loss":float("nan"),"dtype":"auto","bs":8})
    else:
        print("   KHONG lay duoc ket qua. 400 dong cuoi stderr:")
        print((r.stderr or r.stdout)[-1200:])
if sp_secs and dp_secs:
    barchart(["1x T4","2x T4"],[sp_secs,dp_secs],
             "Data parallel qua accelerate launch","giay","{:.1f}s")
    print(f"2 GPU nhanh hon {sp_secs/dp_secs:.2f}x (ly tuong la 2.00x; chenh lech la overhead dong bo)")
    print("Voi model 0.6B va dataset nho, overhead chiem ty trong lon -> loi it hon ky vong.")
    print("Model/dataset cang lon thi ty le cang gan 2x.")

## 7. 📊 Tổng hợp + khuyến nghị

In [ ]:
RESULTS_U=[]; seen=set()
for r in RESULTS:
    if r["name"] not in seen: RESULTS_U.append(r); seen.add(r["name"])
RESULTS_U.sort(key=lambda r:r["secs"])
slow=RESULTS_U[-1]["secs"]
print(f"{'Cau hinh':28s}{'Giay':>8s}{'Nhanh hon x':>13s}{'VRAM GB':>10s}")
print("-"*59)
for r in RESULTS_U:
    v = f"{r['vram']:.2f}" if r['vram']==r['vram'] else "-"
    print(f"{r['name']:28s}{r['secs']:>8.1f}{slow/r['secs']:>12.2f}x{v:>10s}")

top=RESULTS_U[0]
print(f"\nNhanh nhat: {top['name']}  ({top['secs']:.1f}s, nhanh hon cau hinh cham nhat {slow/top['secs']:.2f}x)")

fig,ax=plt.subplots(figsize=(8.2,.46*len(RESULTS_U)+1.6))
vals=[r["secs"] for r in RESULTS_U]; names=[r["name"] for r in RESULTS_U]
cols=[S3 if v==min(vals) else (S2 if v==max(vals) else S1) for v in vals]
ax.barh(names,vals,color=cols,edgecolor=SURFACE,linewidth=2,height=.62)
for i,v in enumerate(vals):
    ax.annotate(f"{v:.1f}s",(v,i),textcoords="offset points",xytext=(5,0),
                va="center",color=INK2,fontsize=9)
ax.invert_yaxis(); ax.grid(axis="x",alpha=.7); ax.set_axisbelow(True)
ax.set_title("Tat ca cau hinh - thoi gian train 1 epoch / 400 mau",loc="left",pad=12,color=INK)
ax.set_xlabel("giay"); plt.tight_layout(); plt.show()

json.dump({"gpu":GPU,"n_gpu":N_GPU,"cap":list(CAP),"bf16_native":BF16_NATIVE,
           "is_bf16_supported_reported":torch.cuda.is_bf16_supported(),
           "results":RESULTS_U}, open("/kaggle/working/speed_report.json","w"), indent=2)
print("\nda luu /kaggle/working/speed_report.json")

### Cách đọc bảng trên

Áp dụng theo thứ tự — cái rẻ nhất trước:

1. **Chọn đúng dtype.** Trên T4 phải `fp16`; `torch.cuda.is_bf16_supported()` nói dối, dùng
   `get_device_capability(0)[0] >= 8`. Đây là lỗi 1 dòng mà tốn nhiều nhất.
2. **Gom mẫu cùng độ dài.** `group_by_length` đã bị bỏ ở transformers v5 → dùng
   `DS.sort("length")`. ⚠️ Sort làm mất tính ngẫu nhiên thứ tự mẫu; với 1 epoch ảnh hưởng nhỏ,
   nhiều epoch thì nên shuffle lại mỗi epoch hoặc dùng `packing` thay thế.
3. **`packing=True`** nếu dùng TRL và mẫu ngắn hơn `max_length` nhiều.
4. **Tăng batch** tới sát trần VRAM.
5. **`attn_implementation="sdpa"`** — đừng phí thời gian với FlashAttention-2 trên T4.
6. **2 GPU** chỉ khi model/dataset đủ lớn để overhead đồng bộ không nuốt hết lợi ích.

### Những thứ KHÔNG đo ở đây nhưng tiết kiệm nhiều thời gian thực tế

- **Cache model thành Kaggle Model/Dataset** và mount vào `/kaggle/input` → khỏi tải lại mỗi session.
- **`SAVE_MERGED=False`** → khỏi upload 1.2GB output cuối run.
- **Dùng "Save Version" (batch)** thay vì session interactive — interactive đốt quota cả khi idle.
- **Unsloth** công bố nhanh ~2x. Cài chỉ 0.4 phút nhưng **ghi đè torch** → phải notebook riêng.

### 🔴 Bài học API — chính notebook này đã chết vì nó

Run đầu tiên ERROR với:
```
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'group_by_length'
```

`transformers` v5 đã bỏ `group_by_length` (giống `warmup_ratio`). **Đừng tin tutorial, hãy introspect:**
```python
import inspect
from transformers import TrainingArguments
P = set(inspect.signature(TrainingArguments.__init__).parameters)
print("group_by_length" in P, "warmup_ratio" in P)
```

Notebook này giờ có helper `ta(**kw)` tự lọc tham số không tồn tại, nên nó **không chết** khi
bạn chạy trên một bản transformers khác. Đó là cách viết code sống được qua các lần đổi API.